In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
from sodapy import Socrata
import geopandas as gpd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_seq_items", None)

BASE_DIR      = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data"
MENSUALES_DIR = os.path.join(BASE_DIR, "Escenarios Cambio Climatico IDEAM IV comunicacion", "Mensuales")
WEB_DATA_DIR  = os.path.join(BASE_DIR, "Scripts Python", "webpage_climate", "data")

os.chdir(BASE_DIR)
print("Directorio de trabajo :", os.getcwd())
print("Carpeta web/data      :", WEB_DATA_DIR)

In [ ]:
pip install geopandas

In [ ]:
pip install sodapy

## Datos diarios – Extracción API y cruce con alertas históricas

In [ ]:
# Carga alertas históricas generadas por datos precipitacion historicos.ipynb
out_estaciones = os.path.join(WEB_DATA_DIR, "alerta_historica_estaciones.csv")
estaciones_alerta = pd.read_csv(out_estaciones)

print(f"Estaciones históricas cargadas: {len(estaciones_alerta):,}")
print("\n=== Distribución de alerta compuesta (histórica) ===")
print(estaciones_alerta['alerta_compuesta'].value_counts())
estaciones_alerta.head()

In [ ]:
# Carga davipola y proyecta a EPSG 3116 (necesario para sjoin con municipios)
davipola = pd.read_excel(os.path.join(MENSUALES_DIR, "davipola_dane.xlsx"))

gdf_mun = gpd.GeoDataFrame(
    davipola,
    geometry=gpd.points_from_xy(davipola.LONGITUD, davipola.LATITUD),
    crs="EPSG:4326",
).to_crs(epsg=3116)

print(f"Municipios cargados: {len(gdf_mun):,}")

## Datos en tiempo real – API IDEAM

In [ ]:
DATASET_ID = "s54a-sgyg"
client = Socrata("www.datos.gov.co", None)
fecha_mapa = client.get(DATASET_ID, select="max(fechaobservacion)")[0]["max_fechaobservacion"][:10]
print(f"Última fecha disponible: {fecha_mapa}")

where = (
    f"fechaobservacion >= '{fecha_mapa}T00:00:00' "
    f"AND fechaobservacion < '{fecha_mapa}T23:59:59.999'"
)

records, offset = [], 0
while True:
    batch = client.get(DATASET_ID, where=where, limit=100_000, offset=offset)
    if not batch:
        break
    records.extend(batch)
    offset += 100_000
    print(f"  {len(records):,} registros descargados...")
client.close()

In [ ]:
# Agrega lecturas a nivel de estación
df_api = pd.DataFrame.from_records(records)

for col in ("valorobservado", "latitud", "longitud"):
    df_api[col] = pd.to_numeric(df_api[col], errors="coerce")

df_api = df_api.dropna(subset=["latitud", "longitud", "valorobservado"])
df_api = df_api[df_api["valorobservado"] >= 0]

STATION_COLS_API = [
    "codigoestacion", "nombreestacion", "departamento",
    "municipio", "zonahidrografica", "latitud", "longitud",
]

df_dia = (
    df_api.groupby(STATION_COLS_API, as_index=False)
    .agg(
        precip_acum_mm=("valorobservado", "sum"),
        precip_max_10min=("valorobservado", "max"),
        n_lecturas=("valorobservado", "count"),
    )
)
del df_api

# Normaliza cod_norm a string en ambos lados antes del merge
df_dia["cod_norm"] = df_dia["codigoestacion"].astype(str).str.strip().str.lstrip("0")
estaciones_alerta["cod_norm"] = estaciones_alerta["cod_norm"].astype(str).str.strip().str.lstrip("0")

# Cruce con alertas históricas — incluye indicadores de sequía y tipo_alerta
cols_merge = [
    "cod_norm", "alerta", "alerta_compuesta",
    "frecuencia_extremos", "frecuencia_reciente", "ratio_reciente", "tendencia",
    "sequia_categoria", "sequia_nivel", "tipo_alerta",
]

df_dia = df_dia.merge(estaciones_alerta[cols_merge], on="cod_norm", how="left")

df_dia["alerta"]              = df_dia["alerta"].fillna(0).astype(int)
df_dia["alerta_compuesta"]    = df_dia["alerta_compuesta"].fillna("BAJA")
df_dia["frecuencia_extremos"] = df_dia["frecuencia_extremos"].fillna(0)
df_dia["frecuencia_reciente"] = df_dia["frecuencia_reciente"].fillna(0)
df_dia["ratio_reciente"]      = df_dia["ratio_reciente"].fillna(np.nan)
df_dia["tendencia"]           = df_dia["tendencia"].fillna("sin_datos")
df_dia["sequia_categoria"]    = df_dia["sequia_categoria"].fillna("NORMAL")
df_dia["sequia_nivel"]        = df_dia["sequia_nivel"].fillna(0).astype(int)
df_dia["tipo_alerta"]         = df_dia["tipo_alerta"].fillna("Sin alerta")

print(f"Estaciones activas hoy     : {len(df_dia):,}")
print(f"  Con alerta CRÍTICA        : {(df_dia['alerta_compuesta'] == 'CRÍTICA').sum():,}")
print(f"  Con alerta ALTA           : {(df_dia['alerta_compuesta'] == 'ALTA').sum():,}")
print(f"  Con alerta MODERADA       : {(df_dia['alerta_compuesta'] == 'MODERADA').sum():,}")
print(f"  Sin match en histórico    : {(df_dia['frecuencia_extremos'] == 0).sum():,}")

In [ ]:
# ── Helpers robustos ante NaN ─────────────────────────────────────────────
def modo_seguro(serie, default='BAJA'):
    vc = serie.dropna().value_counts()
    return vc.idxmax() if not vc.empty else default

def tendencia_muni(serie):
    vals = serie.dropna()
    if vals.empty:
        return 'sin_datos'
    if 'creciente' in vals.values:
        return 'creciente'
    vc = vals.value_counts()
    return vc.idxmax() if not vc.empty else 'sin_datos'

def modo_sequia(serie):
    return modo_seguro(serie, default='NORMAL')

def tipo_alerta_muni(serie):
    vals = serie.dropna()
    if vals.empty:
        return 'Sin alerta'
    if 'Ambos' in vals.values:
        return 'Ambos'
    vc = vals.value_counts()
    return vc.idxmax()

NIVEL_NUMERICO = {'BAJA': 0, 'MODERADA': 1, 'ALTA': 2, 'CRÍTICA': 3}

# Sjoin estaciones del día → municipios
gdf_dia = gpd.GeoDataFrame(
    df_dia,
    geometry=gpd.points_from_xy(df_dia.longitud, df_dia.latitud),
    crs="EPSG:4326",
).to_crs(epsg=3116)

df_muni = gpd.sjoin_nearest(gdf_mun, gdf_dia, how="left", distance_col="dist_m")

# Agrupación a nivel municipal
df_muni = (
    df_muni
    .groupby(["COD_MPIO", "NOM_MPIO", "NOM_DPTO", "LATITUD", "LONGITUD"], as_index=False)
    .agg(
        precip_acum_mm=("precip_acum_mm", "mean"),
        precip_max_10min=("precip_max_10min", "max"),
        n_estaciones=("codigoestacion", "count"),
        alerta_compuesta=("alerta_compuesta", modo_seguro),
        frecuencia_extremos=("frecuencia_extremos", "max"),
        frecuencia_reciente=("frecuencia_reciente", "max"),
        ratio_reciente=("ratio_reciente", "max"),
        tendencia=("tendencia", tendencia_muni),
        sequia_categoria=("sequia_categoria", modo_sequia),
        tipo_alerta=("tipo_alerta", tipo_alerta_muni),
    )
)

# alerta numérico derivado de alerta_compuesta (moda) → consistencia garantizada
df_muni["alerta"] = df_muni["alerta_compuesta"].map(NIVEL_NUMERICO).fillna(0).astype(int)
df_muni["fecha"]  = fecha_mapa

print(f"Municipios con datos   : {len(df_muni):,}")
print(f"\n=== Alerta compuesta por municipio (nivel dominante) ===")
print(df_muni['alerta_compuesta'].value_counts())
print(f"\n=== Tipo de alerta municipal ===")
print(df_muni['tipo_alerta'].value_counts())
print(f"\nTotal con alerta > BAJA: {(df_muni['alerta'] > 0).sum():,}")

out_path = os.path.join(WEB_DATA_DIR, "datos_municipios.csv")
df_muni.to_csv(out_path, index=False, encoding="utf-8-sig")
print("\nGuardado:", out_path)
df_muni.sort_values('alerta', ascending=False).head(20)